# MemScope — Testing Notebook

This notebook can live **anywhere** on your machine or on Kaggle/Colab.
It installs MemScope directly from GitHub, then runs all four leak scenarios.

| Cell | Test | Expected pattern |
|---|---|---|
| 1 | Install + sanity check | — |
| 2 | Leak: computation graph held in list | `linear_leak` |
| 3 | Leak: output tensors stored in list | `linear_leak` |
| 4 | Leak: validation outputs never cleared | `linear_leak` or `periodic_accumulation` |
| 5 | Healthy training loop | `healthy` |
| 6 | Per-layer breakdown with a deeper model | chart in report |
| 7 | Analyzer unit checks | all 3 correct |
| 8 | Report summary | lists all HTML files |

## Cell 1 — Install MemScope from GitHub

In [ ]:
# Installs directly from GitHub — no local clone needed
!pip install git+https://github.com/mohd-ravish/memscope.git --quiet

In [ ]:
import sys, time, random
import torch
import torch.nn as nn
import torch.optim as optim
from memscope import MemoryProfiler, analyze

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
print(f"MemScope: imported OK")

REPORT_DIR = "./memscope-report"  # reports go here, next to this notebook

---
## Cell 2 — Leak: computation graph held in list

**Bug:** `losses.append(loss)` — no `.item()`. Every step the full computation graph
(all intermediate tensors needed for backprop) stays alive in memory.

**Fix:** `losses.append(loss.item())`

**Expected:** `linear_leak`, severity `high`

> Uses a large model (1024×1024, batch 256) so the leak is ~2 MB/step and clearly visible.

In [ ]:
model1 = nn.Linear(1024, 1024).to(device)
losses1 = []

with MemoryProfiler(output_dir=f"{REPORT_DIR}/test1_graph_leak",
                    sample_interval_ms=30,
                    model=model1) as prof:
    for step in range(80):
        x = torch.randn(256, 1024, device=device)
        loss = model1(x).sum()
        losses1.append(loss)   # BUG: no .item() — keeps computation graph alive
        prof.step()
        time.sleep(0.02)

a1 = analyze(prof.sampler.samples)
print(f"Samples  : {len(prof.sampler.samples)}")
print(f"Pattern  : {a1.pattern}")
print(f"Severity : {a1.severity}")
print(f"Growth   : {a1.growth_mb_per_step} MB/step")
print(f"Message  : {a1.message}")

prof.report()

assert a1.pattern in ("linear_leak", "periodic_accumulation"), \
    f"Expected a leak, got: {a1.pattern}"
print("\nPASS")

---
## Cell 3 — Leak: output tensors stored in a list

**Bug:** `results.append(out)` stores the tensor directly — it stays in memory forever.

**Fix:** `results.append(out.detach().cpu())`

**Expected:** `linear_leak`, severity `high`

In [ ]:
model2 = nn.Sequential(
    nn.Linear(1024, 2048),
    nn.ReLU(),
    nn.Linear(2048, 1024),
).to(device)

results2 = []

with MemoryProfiler(output_dir=f"{REPORT_DIR}/test2_tensor_list",
                    sample_interval_ms=30,
                    model=model2) as prof:
    for step in range(80):
        x = torch.randn(256, 1024, device=device)
        out = model2(x)
        results2.append(out)   # BUG: tensor stays in memory forever
        prof.step()
        time.sleep(0.02)

a2 = analyze(prof.sampler.samples)
print(f"Samples  : {len(prof.sampler.samples)}")
print(f"Pattern  : {a2.pattern}")
print(f"Severity : {a2.severity}")
print(f"Growth   : {a2.growth_mb_per_step} MB/step")

prof.report()

assert a2.pattern in ("linear_leak", "periodic_accumulation"), \
    f"Expected a leak, got: {a2.pattern}"
print("\nPASS")

---
## Cell 4 — Leak: validation outputs never cleared

**Bug:** `val_outputs` grows every epoch and is never cleared between epochs.

**Fix:** Add `val_outputs.clear()` after each validation loop.

**Expected:** `linear_leak` or `periodic_accumulation`

In [ ]:
model3 = nn.Linear(1024, 1024).to(device)
val_outputs = []

with MemoryProfiler(output_dir=f"{REPORT_DIR}/test3_periodic",
                    sample_interval_ms=30,
                    model=model3) as prof:
    for epoch in range(20):
        # Training — correct
        for _ in range(4):
            x = torch.randn(128, 1024, device=device)
            model3(x).sum().item()   # .item() releases the graph
            prof.step()
            time.sleep(0.015)

        # Validation — leaky
        with torch.no_grad():
            for _ in range(6):
                x = torch.randn(128, 1024, device=device)
                val_outputs.append(model3(x))   # BUG: never cleared between epochs
                time.sleep(0.015)

        print(f"  epoch {epoch:2d} | val_outputs: {len(val_outputs)} tensors")

a3 = analyze(prof.sampler.samples)
print(f"\nPattern  : {a3.pattern}")
print(f"Severity : {a3.severity}")
print(f"Growth   : {a3.growth_mb_per_step} MB/step")

prof.report()

assert a3.pattern in ("linear_leak", "periodic_accumulation"), \
    f"Expected a leak, got: {a3.pattern}"
print("\nPASS")

---
## Cell 5 — Healthy training loop

A correctly written loop — every tensor released after each step.

**Expected:** `healthy`, severity `none`

In [ ]:
model4 = nn.Linear(512, 512).to(device)
optimizer4 = optim.Adam(model4.parameters(), lr=1e-3)

with MemoryProfiler(output_dir=f"{REPORT_DIR}/test4_healthy",
                    sample_interval_ms=30,
                    model=model4) as prof:
    for step in range(80):
        optimizer4.zero_grad()
        x = torch.randn(64, 512, device=device)
        loss = model4(x).sum()
        loss.backward()
        optimizer4.step()
        loss.item()   # releases the graph
        prof.step()
        time.sleep(0.02)

a4 = analyze(prof.sampler.samples)
print(f"Pattern  : {a4.pattern}")
print(f"Severity : {a4.severity}")
print(f"Growth   : {a4.growth_mb_per_step} MB/step")
print(f"Message  : {a4.message}")

prof.report()

assert a4.pattern == "healthy", \
    f"Expected healthy, got: {a4.pattern}"
print("\nPASS")

---
## Cell 6 — Per-layer memory breakdown

Pass `model=model` to `MemoryProfiler` so forward hooks are registered automatically.
Open the report and check the **Per-Layer Memory Breakdown** bar chart.

In [ ]:
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.bottleneck = nn.Linear(128, 64)
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),  nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784),
        )
    def forward(self, x):
        return self.decoder(self.bottleneck(self.encoder(x)))

deep_model = DeepNet().to(device)
opt5 = optim.Adam(deep_model.parameters(), lr=1e-3)
crit5 = nn.MSELoss()

with MemoryProfiler(output_dir=f"{REPORT_DIR}/test5_layer_breakdown",
                    sample_interval_ms=30,
                    model=deep_model) as prof:
    for step in range(60):
        opt5.zero_grad()
        x = torch.randn(128, 784, device=device)
        out = deep_model(x)
        loss = crit5(out, x)
        loss.backward()
        opt5.step()
        loss.item()
        prof.step()
        time.sleep(0.02)

print("Layers tracked:")
for name, sizes in sorted(prof.hook_manager.stats.items()):
    avg = sum(sizes) / len(sizes)
    print(f"  {name:35s}  avg {avg:.4f} MB")

assert len(prof.hook_manager.stats) > 0, "No layer stats captured"
prof.report()
print("\nPASS")

---
## Cell 7 — Analyzer unit checks

Feeds synthetic sample data directly into the analyzer — no training loop needed.
Quick check that the pattern detection logic is correct.

In [ ]:
from memscope.sampler import MemorySample
from memscope.analyzer import analyze

def make_samples(values):
    return [
        MemorySample(timestamp=float(i), step=i,
                     allocated_mb=v, reserved_mb=v * 1.2, cpu_mb=10.0)
        for i, v in enumerate(values)
    ]

cases = [
    ("linear growth",   [100 + i * 5 for i in range(50)],                   "linear_leak"),
    ("flat / noise",    [400 + random.uniform(-4, 4) for _ in range(50)],   "healthy"),
    ("too few samples", [100.0] * 5,                                          "insufficient_data"),
]

all_pass = True
for label, values, expected in cases:
    result = analyze(make_samples(values))
    ok = result.pattern == expected
    if not ok:
        all_pass = False
    print(f"  {'PASS' if ok else 'FAIL'}  [{label:18s}]  expected={expected:18s}  got={result.pattern}")

assert all_pass, "One or more analyzer checks failed"
print("\nPASS — all analyzer checks correct")

---
## Cell 8 — Report summary

Lists all generated HTML reports. Open any of them in your browser — no server needed.

In [ ]:
import glob, os
reports = sorted(glob.glob(f"{REPORT_DIR}/**/memscope_report.html", recursive=True))
print(f"Generated {len(reports)} reports:\n")
for r in reports:
    print(f"  {os.path.abspath(r)}")